In [1]:
import os
import pandas as pd
import functions as func
import numpy as np
from tqdm import tqdm
from pprint import pprint

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.0' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


### Constants

In [2]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
str_step = os.getcwd().split('/')[-1]
print(f'Step: {str_step}')
str_dirname_output = './output'

str_id = 'uniqueid'
str_datecol = 'applicationdate__app'
str_target = 'target'
flt_threshold = 0.80 # proportion missing in any data set

list_cols_id = [
    str_id,
    str_datecol,
    str_target,
]

Project: 20231010-gen-xii
Step: 02_name


### Output

In [3]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Import column names

In [4]:
%%time

str_filename = 'df_descriptives_train.csv'
str_local_path = f'../../04_eda_pt_2/output/{str_filename}'
list_cols = list(pd.read_csv(str_local_path)['feature'])
# message
print(f'We will be reviewing {len(list_cols)} columns for leaks by name')
print('')

We will be reviewing 2485 columns for leaks by name

CPU times: user 7.68 ms, sys: 3.66 ms, total: 11.3 ms
Wall time: 13.2 ms


### List of Leaky Features by Name

In [5]:
%%time

# get id, date, score, fund columns
list_id_cols, list_date_cols, list_score_cols, list_fund_cols = func.check_columns(list_cols)

# combine lists
list_leaky_by_name = list_id_cols + list_date_cols + list_score_cols + list_fund_cols
# rm dups
list_leaky_by_name = list(dict.fromkeys(list_leaky_by_name))
# message
print('')
print(f'There are {len(list_leaky_by_name)} features with names suggesting they should be dropped')
for a, col in enumerate(list_leaky_by_name):
    print(f'{a+1} - {col}')

print('')

There are 23 columns containing the string "id"
1 - bigaccountid
2 - bigdebtorid
3 - acctid__tu
4 - partyid__tu
5 - permid__tu
6 - middlename__tu
7 - educationevidence__ln
8 - inputprovidedstate__ln
9 - inputprovidedfirstname__ln
10 - inputprovidedlastname__ln
11 - inputprovidedcity__ln
12 - inputprovidedzipcode__ln
13 - inputprovidedssn__ln
14 - inputprovideddateofbirth__ln
15 - inputprovidedphone__ln
16 - inputprovidedlexid__ln
17 - inputprovidedstreetaddress__ln
18 - bigdealertypeid__app
19 - bigdebtorid__app
20 - bigaccountid__app
21 - bigdebtorid__ln
22 - bigaccountid__ln
23 - bigdealerid__app

There are 27 columns containing the strings "date", "dtm", or "dte"
1 - dtmstampcreation__base
2 - dtmapproved__base
3 - dtmdeclined__base
4 - dtmfunded__base
5 - creditasofdate__tu
6 - gnrcrem_frsrpt_dte__base
7 - observationdate__base
8 - open_dte__base
9 - decision_dte__base
10 - observationdate__tu
11 - dtmstampcreation__tu
12 - dateofbirth__tu
13 - chargeoffdate__app
14 - defaultdate__

In [6]:
# remove good cols in list_leaky_by_name
list_dont_drop = [
    'educationevidence__ln',
    'inputprovidedstate__ln',
    'inputprovidedfirstname__ln',
    'inputprovidedlastname__ln',
    'inputprovidedcity__ln',
    'inputprovidedzipcode__ln',
    'inputprovidedssn__ln',
    'inputprovideddateofbirth__ln',
    'inputprovidedphone__ln',
    'inputprovidedlexid__ln',
    'inputprovidedstreetaddress__ln',
    'bigdealertypeid__app',
    'auto_score__ln',
    # 2023-11-06
    'intscore__ln',
    'vantagescore4__tu',
]
list_leaky_by_name = [col for col in list_leaky_by_name if col not in list_dont_drop]

# message
print(f'There are {len(list_leaky_by_name)} features identified as leaky based off of their name')
for a, col in enumerate(list_leaky_by_name):
    print(f'{a+1} - {col}')

There are 51 features identified as leaky based off of their name
1 - bigaccountid
2 - bigdebtorid
3 - acctid__tu
4 - partyid__tu
5 - permid__tu
6 - middlename__tu
7 - bigdebtorid__app
8 - bigaccountid__app
9 - bigdebtorid__ln
10 - bigaccountid__ln
11 - bigdealerid__app
12 - dtmstampcreation__base
13 - dtmapproved__base
14 - dtmdeclined__base
15 - dtmfunded__base
16 - creditasofdate__tu
17 - gnrcrem_frsrpt_dte__base
18 - observationdate__base
19 - open_dte__base
20 - decision_dte__base
21 - observationdate__tu
22 - dtmstampcreation__tu
23 - dateofbirth__tu
24 - chargeoffdate__app
25 - defaultdate__app
26 - dtmfunded__tu
27 - dtmfunded__app
28 - fundeddate__app
29 - dtmapproved__tu
30 - approvaldate__app
31 - dtmapproved__app
32 - dtmdeclined__tu
33 - dtmdeclined__app
34 - tu_creditbureaudate__tu
35 - credit_as_of_date__tu
36 - ssndatelowissued__ln
37 - dtmstampcreation__app
38 - score_cvpropensity__tu
39 - score_epd__tu
40 - score_newaccount__tu
41 - score_bankcard__tu
42 - score_cvaut

### List of Leaky Features Manually

In [7]:
# create list leaky manual 
list_leaky_manual = [
    # obviously bad
    'bitlhmgroup__app',
    'bitfunded__app',
    'dealerzip__app',
    'dealercity__app',
    'analyticsmatchkey__tu',
    'strzipcode__app',
    'bitapproved__app',
    'strcity__app',
    'fltapprovedapr_contract__app',
    'subjectage__ln',
    'applicationmonth__app',
    'applicationdayofweek__app',
    'applicationquarter__app',
    'fltapproveddebttoincome__app',
    'bitsystemdecline__app',
    'addrinputtaxyr__ln',
    'bitrolled__app',
    'addrcurrenttaxyr__ln',
    # probably bad
    'linkb011__tu',
    'fltacquisitionfee__app',
    'flttaxgrossreceipts__app',
    # 24 month time-series features
    'agg101__tu',
    'agg102__tu',
    'agg103__tu',
    'agg104__tu',
    'agg105__tu',
    'agg106__tu',
    'agg107__tu',
    'agg108__tu',
    'agg109__tu',
    'agg110__tu',
    'agg111__tu',
    'agg112__tu',
    'agg113__tu',
    'agg114__tu',
    'agg115__tu',
    'agg116__tu',
    'agg117__tu',
    'agg118__tu',
    'agg119__tu',
    'agg120__tu',
    'agg121__tu',
    'agg122__tu',
    'agg123__tu',
    'agg124__tu',
    'agg201__tu',
    'agg202__tu',
    'agg203__tu',
    'agg204__tu',
    'agg205__tu',
    'agg206__tu',
    'agg207__tu',
    'agg208__tu',
    'agg209__tu',
    'agg210__tu',
    'agg211__tu',
    'agg212__tu',
    'agg213__tu',
    'agg214__tu',
    'agg215__tu',
    'agg216__tu',
    'agg217__tu',
    'agg218__tu',
    'agg219__tu',
    'agg220__tu',
    'agg221__tu',
    'agg222__tu',
    'agg223__tu',
    'agg224__tu',
    'agg301__tu',
    'agg302__tu',
    'agg303__tu',
    'agg304__tu',
    'agg305__tu',
    'agg306__tu',
    'agg307__tu',
    'agg308__tu',
    'agg309__tu',
    'agg310__tu',
    'agg311__tu',
    'agg312__tu',
    'agg313__tu',
    'agg314__tu',
    'agg315__tu',
    'agg316__tu',
    'agg317__tu',
    'agg318__tu',
    'agg319__tu',
    'agg320__tu',
    'agg321__tu',
    'agg322__tu',
    'agg323__tu',
    'agg324__tu',
    'agg401__tu',
    'agg402__tu',
    'agg403__tu',
    'agg404__tu',
    'agg405__tu',
    'agg406__tu',
    'agg407__tu',
    'agg408__tu',
    'agg409__tu',
    'agg410__tu',
    'agg411__tu',
    'agg412__tu',
    'agg413__tu',
    'agg414__tu',
    'agg415__tu',
    'agg416__tu',
    'agg417__tu',
    'agg418__tu',
    'agg419__tu',
    'agg420__tu',
    'agg421__tu',
    'agg422__tu',
    'agg423__tu',
    'agg424__tu',
    'agg501__tu',
    'agg502__tu',
    'agg503__tu',
    'agg504__tu',
    'agg505__tu',
    'agg506__tu',
    'agg507__tu',
    'agg508__tu',
    'agg509__tu',
    'agg510__tu',
    'agg511__tu',
    'agg512__tu',
    'agg513__tu',
    'agg514__tu',
    'agg515__tu',
    'agg516__tu',
    'agg517__tu',
    'agg518__tu',
    'agg519__tu',
    'agg520__tu',
    'agg521__tu',
    'agg522__tu',
    'agg523__tu',
    'agg524__tu',
    'agg601__tu',
    'agg602__tu',
    'agg603__tu',
    'agg604__tu',
    'agg605__tu',
    'agg606__tu',
    'agg607__tu',
    'agg608__tu',
    'agg609__tu',
    'agg610__tu',
    'agg611__tu',
    'agg612__tu',
    'agg613__tu',
    'agg614__tu',
    'agg615__tu',
    'agg616__tu',
    'agg617__tu',
    'agg618__tu',
    'agg619__tu',
    'agg620__tu',
    'agg621__tu',
    'agg622__tu',
    'agg623__tu',
    'agg624__tu',
    'agg701__tu',
    'agg702__tu',
    'agg703__tu',
    'agg704__tu',
    'agg705__tu',
    'agg706__tu',
    'agg707__tu',
    'agg708__tu',
    'agg709__tu',
    'agg710__tu',
    'agg711__tu',
    'agg712__tu',
    'agg713__tu',
    'agg714__tu',
    'agg715__tu',
    'agg716__tu',
    'agg717__tu',
    'agg718__tu',
    'agg719__tu',
    'agg720__tu',
    'agg721__tu',
    'agg722__tu',
    'agg723__tu',
    'agg724__tu',
    'agg801__tu',
    'agg802__tu',
    'agg803__tu',
    'agg804__tu',
    'agg805__tu',
    'agg806__tu',
    'agg807__tu',
    'agg808__tu',
    'agg809__tu',
    'agg810__tu',
    'agg811__tu',
    'agg812__tu',
    'agg813__tu',
    'agg814__tu',
    'agg815__tu',
    'agg816__tu',
    'agg817__tu',
    'agg818__tu',
    'agg819__tu',
    'agg820__tu',
    'agg821__tu',
    'agg822__tu',
    'agg823__tu',
    'agg824__tu',
    'aggs101__tu',
    'aggs102__tu',
    'aggs103__tu',
    'aggs104__tu',
    'aggs105__tu',
    'aggs106__tu',
    'aggs107__tu',
    'aggs108__tu',
    'aggs109__tu',
    'aggs110__tu',
    'aggs111__tu',
    'aggs112__tu',
    'aggs113__tu',
    'aggs114__tu',
    'aggs115__tu',
    'aggs116__tu',
    'aggs117__tu',
    'aggs118__tu',
    'aggs119__tu',
    'aggs120__tu',
    'aggs121__tu',
    'aggs122__tu',
    'aggs123__tu',
    'aggs124__tu',
    'rets101__tu',
    'rets102__tu',
    'rets103__tu',
    'rets104__tu',
    'rets105__tu',
    'rets106__tu',
    'rets107__tu',
    'rets108__tu',
    'rets109__tu',
    'rets110__tu',
    'rets111__tu',
    'rets112__tu',
    'rets113__tu',
    'rets114__tu',
    'rets115__tu',
    'rets116__tu',
    'rets117__tu',
    'rets118__tu',
    'rets119__tu',
    'rets120__tu',
    'rets121__tu',
    'rets122__tu',
    'rets123__tu',
    'rets124__tu',
    'revs101__tu',
    'revs102__tu',
    'revs103__tu',
    'revs104__tu',
    'revs105__tu',
    'revs106__tu',
    'revs107__tu',
    'revs108__tu',
    'revs109__tu',
    'revs110__tu',
    'revs111__tu',
    'revs112__tu',
    'revs113__tu',
    'revs114__tu',
    'revs115__tu',
    'revs116__tu',
    'revs117__tu',
    'revs118__tu',
    'revs119__tu',
    'revs120__tu',
    'revs121__tu',
    'revs122__tu',
    'revs123__tu',
    'revs124__tu',
    # 2024-05-07
    'strvehicletype__app', # unavailable at time of application
]

In [8]:
# check if cols in dataset
list_leaky_manual = [col for col in list_leaky_manual if col in list_cols]
print(f'There are {len(list_leaky_manual)} features to remove manually')
for a, col in enumerate(list_leaky_manual):
    print(f'{a+1} - {col}')

There are 285 features to remove manually
1 - bitlhmgroup__app
2 - bitfunded__app
3 - dealerzip__app
4 - dealercity__app
5 - analyticsmatchkey__tu
6 - strzipcode__app
7 - bitapproved__app
8 - strcity__app
9 - fltapprovedapr_contract__app
10 - subjectage__ln
11 - applicationmonth__app
12 - applicationdayofweek__app
13 - applicationquarter__app
14 - fltapproveddebttoincome__app
15 - bitsystemdecline__app
16 - addrinputtaxyr__ln
17 - bitrolled__app
18 - addrcurrenttaxyr__ln
19 - linkb011__tu
20 - fltacquisitionfee__app
21 - flttaxgrossreceipts__app
22 - agg101__tu
23 - agg102__tu
24 - agg103__tu
25 - agg104__tu
26 - agg105__tu
27 - agg106__tu
28 - agg107__tu
29 - agg108__tu
30 - agg109__tu
31 - agg110__tu
32 - agg111__tu
33 - agg112__tu
34 - agg113__tu
35 - agg114__tu
36 - agg115__tu
37 - agg116__tu
38 - agg117__tu
39 - agg118__tu
40 - agg119__tu
41 - agg120__tu
42 - agg121__tu
43 - agg122__tu
44 - agg123__tu
45 - agg124__tu
46 - agg201__tu
47 - agg202__tu
48 - agg203__tu
49 - agg204__tu


### Combine All Leaky Features and Remove Duplicates

In [9]:
# combine lists
list_cols_all = list_leaky_by_name + list_leaky_manual
list_cols_all = list(dict.fromkeys(list_cols_all))

# make df
df = pd.DataFrame({'feature': list_cols_all})

# save
str_filename = 'df_cols_drop.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_csv(str_local_path, index=False)

# show
df

,feature
0,bigaccountid
1,bigdebtorid
2,acctid__tu
3,partyid__tu
4,permid__tu
...,...
330,revs120__tu
331,revs121__tu
332,revs122__tu
333,revs123__tu


### Save ```df_no_leaks.csv```

In [10]:
# create list of cols to keep
list_cols_keep = [col for col in list_cols if col not in list_cols_all]
# extend
list_cols_keep.extend(list_cols_id)
# rm dups
list_cols_keep = list(dict.fromkeys(list_cols_keep))

# create df
df = pd.DataFrame({'feature': list_cols_keep})
# save
str_filename = 'df_cols_noleaks.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_csv(str_local_path, index=False)

# show
df

,feature
0,bitdebtor__base
1,bkc224__tu
2,bkc222__tu
3,bkc205__tu
4,bkc204__tu
...,...
2148,bitdealerapplicantsamestate__app
2149,fltinsuredunemploymentpremium__app
2150,uniqueid
2151,applicationdate__app
